# Rope Flow — Full Pipeline V02
## Data-Driven Pattern Classification (Overhand Detection)

**Course:** MECH 798M / EECE 798K — Data-Driven Modeling for Science & Engineering  
**Input:** Processed CSV files (Device 0 + Device 1) from Stage 1 preprocessing  
**Output:** Per-cycle classification as `overhand` or `unknown`

### Pipeline architecture (DDM methods)

| Stage | DDM method | Dim | Role |
|-------|-----------|-----|------|
| 1 | Cycle detection | — | ||ω|| peak finding with physical prominence threshold |
| 2 | **Fourier decomposition** (Pillar 5: Time series) | 16D | Spectral fingerprint: dominant freq, harmonic ratios, spectral entropy |
| 3 | **Physics features** (Pillar 6: Scientific modeling) | 12D | Period, symmetry, phase lag, jerk, energy ratio, periodicity |
| 4 | **SVD** (Pillar 4: Dimensionality reduction) | 8D | Singular value ratios + effective rank of 12-channel cycle matrix |
| 5 | **Template correlation** | 2D | Phase-aligned similarity to averaged overhand template |
| 6 | **One-class SVM** (Pillar 1: Learning framework) | — | Learns overhand boundary, extensible to reject non-overhand |

**Total:** 38D feature vector per cycle — all per-cycle, no session-broadcast constants.

---
## 0. Setup & configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks, welch, correlate, butter, filtfilt
from scipy.linalg import svd
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────
# Adjust these to your data location
D0_PATH = 'data/processed/20260303_174948_overhand_oli_device0_processed.csv'
D1_PATH = 'data/processed/20260303_174948_overhand_oli_device1_processed.csv'

# ── Configuration ────────────────────────────────────────────
CONFIG = {
    'FS': 50.0,                        # sampling rate [Hz]
    'CYCLE_PROMINENCE_DEGS': 100.0,    # ||ω|| peak prominence [deg/s]
    'CYCLE_MIN_PERIOD_S': 0.5,         # minimum cycle duration [s]
    'CYCLE_MAX_PERIOD_S': 3.0,         # maximum cycle duration [s]
    'TARGET_LEN': 64,                  # resampled cycle length for templates
    'N_FOURIER_HARMONICS': 5,          # harmonics to extract per axis
    'PCA_COMPONENTS': 6,               # matches 99% variance from SVD analysis
    'TEMPLATE_CORR_THRESHOLD': 0.60,   # minimum correlation to match template
    'OCSVM_NU': 0.1,                   # one-class SVM outlier fraction
    'OCSVM_GAMMA': 'scale',
    'MIN_CYCLE_SAMPLES': 10,           # reject cycles shorter than this
}

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
PAL = {'overhand': '#7F77DD', 'unknown': '#E24B4A',
       'dev0': '#5DCAA5', 'dev1': '#7F77DD'}

---
## 1. Data loading

In [ ]:
def load_session(path_d0, path_d1):
    """Load a pair of processed device CSVs."""
    d0 = pd.read_csv(path_d0)
    d1 = pd.read_csv(path_d1)
    return d0, d1

def extract_signals(df):
    """Extract time, acceleration, angular velocity. Gyro: deg/s → rad/s."""
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w', 'ay_w', 'az_w']].values
    omega = df[['gx', 'gy', 'gz']].values * (np.pi / 180.0)
    Q = df[['qw', 'qx', 'qy', 'qz']].values
    return t, Q, A, omega

d0, d1 = load_session(D0_PATH, D1_PATH)
t0, Q0, A0, om0 = extract_signals(d0)
t1, Q1, A1, om1 = extract_signals(d1)
print(f'Loaded: {len(d0)} samples, {t0[-1]:.1f} s duration, {CONFIG["FS"]:.0f} Hz')
print(f'Columns: {list(d0.columns)}')

---
## 2. Cycle detection

Detect rope-flow cycles from ||ω(t)|| peaks using SavGol smoothing + physical prominence threshold.

**Prominence = 100 deg/s** — minimum angular velocity peak for a complete rope swing. Boundaries placed at midpoints between consecutive peaks.

In [ ]:
def detect_cycles(t, omega, fs=50.0):
    mag = np.linalg.norm(omega, axis=1)
    mag_smooth = savgol_filter(mag, window_length=15, polyorder=3)
    prom_rads = CONFIG['CYCLE_PROMINENCE_DEGS'] * np.pi / 180.0
    min_dist = int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)
    peaks, _ = find_peaks(mag_smooth, distance=min_dist, prominence=prom_rads)
    if len(peaks) < 2:
        return [], mag_smooth, peaks
    bounds = ([0]
              + [(peaks[i] + peaks[i+1]) // 2 for i in range(len(peaks) - 1)]
              + [len(t) - 1])
    cycles = [(bounds[i], bounds[i+1]) for i in range(len(bounds) - 1)]
    valid = []
    for s, e in cycles:
        dur = t[e] - t[s]
        if (dur >= CONFIG['CYCLE_MIN_PERIOD_S'] and
            dur <= CONFIG['CYCLE_MAX_PERIOD_S'] and
            (e - s) >= CONFIG['MIN_CYCLE_SAMPLES']):
            valid.append((s, e))
    return valid, mag_smooth, peaks

def pair_cycles(t0, cyc0, t1, cyc1):
    """Match D0/D1 cycles by maximum time overlap. Each D1 cycle used at most once."""
    def overlap(s0, e0, s1, e1):
        return max(0.0, min(t0[e0], t1[e1]) - max(t0[s0], t1[s1]))
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_idx, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used:
                continue
            ov = overlap(c0[0], c0[1], c1[0], c1[1])
            if ov > best_ov:
                best_ov, best_idx = ov, i
        if best_idx >= 0 and best_ov > 0:
            paired0.append(c0)
            paired1.append(cyc1[best_idx])
            used.add(best_idx)
    return paired0, paired1

# Run cycle detection
fs = CONFIG['FS']
cyc0, mag0_smooth, peaks0 = detect_cycles(t0, om0, fs)
cyc1, mag1_smooth, peaks1 = detect_cycles(t1, om1, fs)
paired0, paired1 = pair_cycles(t0, cyc0, t1, cyc1)
print(f'Cycles: D0={len(cyc0)}, D1={len(cyc1)}, paired={len(paired0)}')

In [ ]:
# Diagnostic: cycle detection plot
fig, ax = plt.subplots(figsize=(14, 3))
t_plot = np.arange(len(mag0_smooth)) / fs
ax.plot(t_plot, mag0_smooth, 'k-', lw=0.5, alpha=0.7, label='||ω|| smoothed')
valid_peaks = [p for p in peaks0 if p < len(t_plot)]
ax.plot([t_plot[p] for p in valid_peaks],
        [mag0_smooth[p] for p in valid_peaks],
        'rv', ms=4, alpha=0.7, label=f'{len(valid_peaks)} peaks')
# Shade paired cycles
for i, (s, e) in enumerate(paired0[:10]):
    ax.axvspan(t0[s], t0[e], alpha=0.1, color='#7F77DD')
ax.set_xlabel('Time [s]')
ax.set_ylabel('||ω|| [rad/s]')
ax.set_title(f'Cycle detection — Device 0 (first 10 cycles shaded)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Cycle period distribution
periods = [t0[e] - t0[s] for s, e in paired0]
print(f'Period: mean={np.mean(periods):.3f}±{np.std(periods):.3f} s '
      f'({1/np.mean(periods):.2f} Hz)')

---
## 3. Fourier decomposition — spectral fingerprint per cycle

Extract from ||ω|| and ||a|| per cycle:
- **Dominant frequency** f₀ [Hz]
- **Spectral centroid** — center of mass of PSD
- **Spectral entropy** — Shannon entropy of normalized PSD (regularity measure)
- **Harmonic ratios** — power at f₀, 2f₀, ..., 5f₀ relative to total (overtone structure)

Total: **16D** per cycle (8 features × 2 channels)

In [ ]:
def fourier_features(signal_segment, fs, n_harmonics=5):
    N = len(signal_segment)
    if N < 8:
        return np.zeros(3 + n_harmonics)
    sig = signal_segment - np.mean(signal_segment)
    F = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(N, 1.0 / fs)
    F[0] = 0
    power = F ** 2
    total_power = np.sum(power) + 1e-12
    dom_idx = np.argmax(F[1:]) + 1
    dom_freq = freqs[dom_idx]
    spectral_centroid = np.sum(freqs * power) / total_power
    f0 = dom_freq
    harmonic_ratios = np.zeros(n_harmonics)
    for h in range(n_harmonics):
        target_freq = f0 * (h + 1)
        if target_freq > freqs[-1]:
            break
        idx = np.argmin(np.abs(freqs - target_freq))
        lo = max(1, idx - 1)
        hi = min(len(F), idx + 2)
        harmonic_ratios[h] = np.sum(power[lo:hi]) / total_power
    p_norm = power[1:] / (np.sum(power[1:]) + 1e-12)
    p_norm = p_norm[p_norm > 0]
    spectral_entropy = -np.sum(p_norm * np.log2(p_norm + 1e-12))
    return np.concatenate([[dom_freq, spectral_centroid, spectral_entropy],
                           harmonic_ratios])

def cycle_fourier_features(A, omega, s, e, fs):
    nh = CONFIG['N_FOURIER_HARMONICS']
    omega_mag = np.linalg.norm(omega[s:e], axis=1)
    acc_mag = np.linalg.norm(A[s:e], axis=1)
    return np.concatenate([fourier_features(omega_mag, fs, nh),
                           fourier_features(acc_mag, fs, nh)])

FOURIER_DIM = 2 * (3 + CONFIG['N_FOURIER_HARMONICS'])
FOURIER_NAMES = []
for ch in ['omega_mag', 'acc_mag']:
    FOURIER_NAMES += [f'{ch}_dom_freq', f'{ch}_spec_centroid', f'{ch}_spec_entropy']
    FOURIER_NAMES += [f'{ch}_harm_{i}' for i in range(CONFIG['N_FOURIER_HARMONICS'])]

print(f'Fourier feature dim: {FOURIER_DIM}')

---
## 4. Physics-grounded features

12D feature vector per paired cycle — all physically motivated:

| # | Feature | Physical meaning |
|---|---------|-----------------|
| 1 | `period_s` | Cycle duration |
| 2 | `peak_omega` | Max ||ω|| across both hands |
| 3–4 | `mean_omega_d0/d1` | Mean angular speed per hand |
| 5–6 | `acc_rms_d0/d1` | RMS acceleration per hand |
| 7 | `acc_asymmetry` | Left/right acceleration ratio |
| 8 | `omega_std_ratio` | Left/right ω variability ratio |
| 9 | `phase_lag_ms` | Cross-correlation lag between hands |
| 10 | `jerk_rms` | RMS of da/dt (motion smoothness) |
| 11 | `ke_ratio` | Rotational / total kinetic energy proxy |
| 12 | `periodicity_score` | Autocorrelation peak strength |

In [ ]:
def physics_features(t0, t1, A0, A1, om0, om1, s0, e0, s1, e1, fs):
    tc0, tc1 = t0[s0:e0], t1[s1:e1]
    a0, a1 = A0[s0:e0], A1[s1:e1]
    w0, w1 = om0[s0:e0], om1[s1:e1]
    period = tc0[-1] - tc0[0]
    m0 = np.linalg.norm(w0, axis=1)
    m1 = np.linalg.norm(w1, axis=1)
    peak_omega = max(np.max(m0), np.max(m1))
    mean_omega_d0 = np.mean(m0)
    mean_omega_d1 = np.mean(m1)
    acc_rms_d0 = np.sqrt(np.mean(np.sum(a0**2, axis=1)))
    acc_rms_d1 = np.sqrt(np.mean(np.sum(a1**2, axis=1)))
    acc_asym = acc_rms_d0 / (acc_rms_d1 + 1e-8)
    omega_ratio = np.std(m0) / (np.std(m1) + 1e-8)
    n_min = min(len(m0), len(m1))
    if n_min > 4:
        c0 = m0[:n_min] - m0[:n_min].mean()
        c1 = m1[:n_min] - m1[:n_min].mean()
        corr = np.correlate(c0, c1, mode='full')
        lag_samples = np.argmax(corr) - (n_min - 1)
        phase_lag = lag_samples / fs * 1000
    else:
        phase_lag = 0.0
    if len(a0) > 2:
        jerk = np.diff(a0, axis=0) * fs
        jerk_rms = np.sqrt(np.mean(np.sum(jerk**2, axis=1)))
    else:
        jerk_rms = 0.0
    e_rot = np.mean(m0**2) + np.mean(m1**2)
    e_lin = np.mean(np.sum(a0**2, axis=1)) + np.mean(np.sum(a1**2, axis=1))
    ke_ratio = e_rot / (e_rot + e_lin + 1e-8)
    if len(m0) > 10:
        m0c = m0 - m0.mean()
        ac = np.correlate(m0c, m0c, mode='full')
        ac = ac[len(m0c)-1:]
        ac /= (ac[0] + 1e-12)
        min_lag = max(2, int(0.3 * fs))
        peaks_ac, props = find_peaks(ac[min_lag:], height=0)
        periodicity = props['peak_heights'][0] if len(peaks_ac) > 0 else 0.0
    else:
        periodicity = 0.0
    return np.array([
        period, peak_omega, mean_omega_d0, mean_omega_d1,
        acc_rms_d0, acc_rms_d1, acc_asym, omega_ratio,
        phase_lag, jerk_rms, ke_ratio, periodicity
    ])

PHYSICS_NAMES = [
    'period_s', 'peak_omega', 'mean_omega_d0', 'mean_omega_d1',
    'acc_rms_d0', 'acc_rms_d1', 'acc_asymmetry', 'omega_std_ratio',
    'phase_lag_ms', 'jerk_rms', 'ke_ratio', 'periodicity_score'
]
print(f'Physics feature dim: {len(PHYSICS_NAMES)}')

---
## 5. SVD — low-rank structure of multi-channel signal

Each paired cycle is resampled to a fixed (12, 64) matrix:
- 12 channels: [a₀_xyz, ω₀_xyz, a₁_xyz, ω₁_xyz]
- 64 time steps (uniform resampling)

**SVD** of this matrix reveals the effective dimensionality of the motion.
From the global SVD analysis: **5–6 modes capture 99% variance** for overhand, confirming it is a low-rank cyclic motion.

Features extracted:
- **Singular value ratios** σᵢ/σ₀ for i=1..5 (shape of the spectrum)
- **Effective rank** (# of σ > 0.01·σ_max)
- **Total energy** Σσ² (overall motion intensity)

In [ ]:
def resample_cycle(signal, target_len):
    n = len(signal)
    if n < 2:
        return np.zeros(target_len) if signal.ndim == 1 else np.zeros((target_len, signal.shape[1]))
    x_old = np.linspace(0, 1, n)
    x_new = np.linspace(0, 1, target_len)
    if signal.ndim == 1:
        return np.interp(x_new, x_old, signal)
    else:
        return np.column_stack([np.interp(x_new, x_old, signal[:, j])
                                for j in range(signal.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1, target_len=64):
    state0 = np.column_stack([A0[s0:e0], om0[s0:e0]])
    state1 = np.column_stack([A1[s1:e1], om1[s1:e1]])
    r0 = resample_cycle(state0, target_len)
    r1 = resample_cycle(state1, target_len)
    return np.column_stack([r0, r1]).T  # (12, target_len)

def compute_svd_features(cycle_matrix, n_components=6):
    U, S, Vt = svd(cycle_matrix, full_matrices=False)
    S_ratio = S[:n_components] / (S[0] + 1e-12)
    eff_rank = np.sum(S > 0.01 * S[0])
    total_energy = np.sum(S**2)
    return np.concatenate([S_ratio, [eff_rank, total_energy]])

SVD_DIM = CONFIG['PCA_COMPONENTS'] + 2
SVD_NAMES = ([f'sv_ratio_{i}' for i in range(CONFIG['PCA_COMPONENTS'])]
             + ['eff_rank', 'total_energy'])
print(f'SVD feature dim: {SVD_DIM}')

# Build all cycle matrices
cycle_matrices = []
for (s0, e0), (s1, e1) in zip(paired0, paired1):
    cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1, CONFIG['TARGET_LEN'])
    cycle_matrices.append(cm)
print(f'Built {len(cycle_matrices)} cycle matrices of shape {cycle_matrices[0].shape}')

In [ ]:
# Diagnostic: SVD spectrum of a representative cycle
mid = len(cycle_matrices) // 2
U, S, Vt = svd(cycle_matrices[mid], full_matrices=False)
cumvar = np.cumsum(S**2) / np.sum(S**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(len(S)), S, color='#7F77DD', alpha=0.8)
ax1.set_xlabel('Mode index')
ax1.set_ylabel('Singular value σ')
ax1.set_title('SVD spectrum (representative cycle)')

ax2.plot(range(len(cumvar)), cumvar * 100, 'o-', color='#5DCAA5')
ax2.axhline(99, color='gray', ls='--', lw=1, label='99%')
ax2.axhline(95, color='gray', ls=':', lw=1, label='95%')
ax2.set_xlabel('Number of modes')
ax2.set_ylabel('Cumulative variance [%]')
ax2.set_title('Cumulative explained variance')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

r99 = int(np.searchsorted(cumvar, 0.99)) + 1
print(f'Modes for 99% variance: {r99}')

---
## 6. Template correlation — overhand signature matching

Build a **phase-aligned** overhand template:
1. Start from the median-energy cycle as initial reference
2. Circularly shift all cycles to maximize correlation on ωx channel
3. Average the IQR-filtered aligned cycles (2 iterations)

Each new cycle is scored by **normalized correlation** (mean + min across 12 channels) against the template.

In [ ]:
def phase_align(cycle_matrix, reference, ref_channel=3):
    """Circularly shift to maximize correlation on ref_channel (default: ωx of D0)."""
    x = cycle_matrix[ref_channel]
    y = reference[ref_channel]
    x_c = x - np.mean(x)
    y_c = y - np.mean(y)
    corr = np.correlate(np.tile(x_c, 2), y_c, mode='valid')
    shift = np.argmax(corr)
    if shift >= len(x):
        shift -= len(x)
    return np.roll(cycle_matrix, -shift, axis=1), shift

def build_template(all_cycle_matrices):
    energies = np.array([np.sum(m**2) for m in all_cycle_matrices])
    median_idx = np.argmin(np.abs(energies - np.median(energies)))
    template = all_cycle_matrices[median_idx].copy()
    for iteration in range(2):
        aligned = [phase_align(cm, template)[0] for cm in all_cycle_matrices]
        q25, q75 = np.percentile(energies, [25, 75])
        mask = (energies >= q25) & (energies <= q75)
        selected = [a for a, keep in zip(aligned, mask) if keep]
        if len(selected) < 3:
            selected = aligned
        template = np.mean(selected, axis=0)
    return template

def template_correlation(cycle_matrix, template):
    aligned, _ = phase_align(cycle_matrix, template)
    n_channels = aligned.shape[0]
    corrs = np.zeros(n_channels)
    for ch in range(n_channels):
        x = aligned[ch] - np.mean(aligned[ch])
        y = template[ch] - np.mean(template[ch])
        denom = (np.linalg.norm(x) * np.linalg.norm(y) + 1e-12)
        corrs[ch] = np.dot(x, y) / denom
    return np.mean(corrs), np.min(corrs)

TEMPLATE_NAMES = ['template_corr_mean', 'template_corr_min']

# Build template
template = build_template(cycle_matrices)
print(f'Template shape: {template.shape}')

In [ ]:
# Diagnostic: template visualization
channel_names = ['ax_d0','ay_d0','az_d0','ωx_d0','ωy_d0','ωz_d0',
                 'ax_d1','ay_d1','az_d1','ωx_d1','ωy_d1','ωz_d1']
fig, axes = plt.subplots(3, 4, figsize=(16, 8), sharex=True)
x = np.linspace(0, 1, template.shape[1])
for i, (ax, name) in enumerate(zip(axes.flat, channel_names)):
    ax.plot(x, template[i], color='#7F77DD', lw=2, label='template')
    # Overlay 5 random aligned cycles
    for j in np.random.choice(len(cycle_matrices), min(5, len(cycle_matrices)), replace=False):
        aligned, _ = phase_align(cycle_matrices[j], template)
        ax.plot(x, aligned[i], color='gray', lw=0.3, alpha=0.5)
    ax.set_title(name, fontsize=9)
    ax.set_ylabel('')
axes[2, 0].set_xlabel('Phase')
axes[2, 1].set_xlabel('Phase')
axes[2, 2].set_xlabel('Phase')
axes[2, 3].set_xlabel('Phase')
plt.suptitle('Overhand template (purple) vs individual cycles (gray)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Feature assembly

In [ ]:
ALL_FEAT_NAMES = PHYSICS_NAMES + FOURIER_NAMES + SVD_NAMES + TEMPLATE_NAMES
N_FEAT = len(ALL_FEAT_NAMES)

def extract_all_features(t0, t1, A0, A1, om0, om1,
                         paired0, paired1, template, fs):
    features = []
    for (s0, e0), (s1, e1) in zip(paired0, paired1):
        phys = physics_features(t0, t1, A0, A1, om0, om1, s0, e0, s1, e1, fs)
        fourier = cycle_fourier_features(A0, om0, s0, e0, fs)
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1, CONFIG['TARGET_LEN'])
        svd_feat = compute_svd_features(cm, CONFIG['PCA_COMPONENTS'])
        corr_mean, corr_min = template_correlation(cm, template)
        tmpl_feat = np.array([corr_mean, corr_min])
        feat = np.concatenate([phys, fourier, svd_feat, tmpl_feat])
        features.append(feat)
    return np.array(features)

X = extract_all_features(t0, t1, A0, A1, om0, om1,
                         paired0, paired1, template, fs)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
print(f'Feature matrix: {X.shape[0]} cycles × {X.shape[1]} features')
print(f'  Physics:  {len(PHYSICS_NAMES)}D')
print(f'  Fourier:  {FOURIER_DIM}D')
print(f'  SVD:      {SVD_DIM}D')
print(f'  Template: {len(TEMPLATE_NAMES)}D')

In [ ]:
# Feature distribution heatmap
fig, ax = plt.subplots(figsize=(16, 6))
X_s = StandardScaler().fit_transform(X)
im = ax.imshow(X_s.T, aspect='auto', cmap='RdBu_r', vmin=-3, vmax=3)
ax.set_xlabel('Cycle index')
ax.set_ylabel('Feature')
ax.set_yticks(range(N_FEAT))
ax.set_yticklabels(ALL_FEAT_NAMES, fontsize=6)
plt.colorbar(im, ax=ax, label='Standardized value')
ax.set_title(f'Feature matrix ({X.shape[0]} cycles × {X.shape[1]} features)')
plt.tight_layout()
plt.show()

---
## 8. Classification — overhand detection

**Two-stage approach:**

1. **One-class SVM** learns the boundary of the overhand feature distribution (ν=0.1 → expects ~10% outliers)
2. **Session-level vote** — if >50% of cycles pass the strict test, the session is classified as overhand and the threshold is relaxed to reject only the bottom 2% (extreme outliers like warmup/transition cycles)

**Confidence score** = 60% SVM decision + 40% template correlation (normalized)

In [ ]:
def train_overhand_detector(X_features):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X_features)
    ocsvm = OneClassSVM(kernel='rbf', nu=CONFIG['OCSVM_NU'], gamma=CONFIG['OCSVM_GAMMA'])
    ocsvm.fit(X_s)
    stats = {
        'mean': np.mean(X_features, axis=0),
        'std': np.std(X_features, axis=0),
        'q25': np.percentile(X_features, 25, axis=0),
        'q75': np.percentile(X_features, 75, axis=0),
    }
    return ocsvm, scaler, stats

def classify_cycles(X_features, ocsvm, scaler, template_corr_idx):
    X_s = scaler.transform(X_features)
    svm_decision = ocsvm.decision_function(X_s)
    corr_mean = X_features[:, template_corr_idx]
    # Normalize to [0, 1]
    svm_norm = (svm_decision - np.min(svm_decision)) / (np.ptp(svm_decision) + 1e-12)
    corr_norm = (corr_mean - np.min(corr_mean)) / (np.ptp(corr_mean) + 1e-12)
    confidence = 0.6 * svm_norm + 0.4 * corr_norm
    # Stage 1: strict — bottom 5% are outliers
    strict_threshold = np.percentile(confidence, 5)
    initial_labels = ['overhand' if c > strict_threshold else 'unknown' for c in confidence]
    # Stage 2: session vote — if majority overhand, relax to bottom 2%
    n_oh = sum(1 for l in initial_labels if l == 'overhand')
    if (n_oh / len(initial_labels)) > 0.5:
        relax_threshold = np.percentile(confidence, 2)
        labels = ['overhand' if c > relax_threshold else 'unknown' for c in confidence]
    else:
        labels = initial_labels
    return labels, confidence.tolist()

# Train and classify
ocsvm, scaler, stats = train_overhand_detector(X)
template_corr_idx = ALL_FEAT_NAMES.index('template_corr_mean')
labels, confidences = classify_cycles(X, ocsvm, scaler, template_corr_idx)

n_overhand = sum(1 for l in labels if l == 'overhand')
accuracy = n_overhand / len(labels) * 100

print(f'\n{"="*60}')
print(f'CLASSIFICATION RESULT')
print(f'{"="*60}')
print(f'  Ground truth:     overhand')
print(f'  Total cycles:     {len(labels)}')
print(f'  Detected overhand: {n_overhand} ({accuracy:.1f}%)')
print(f'  Unknown:           {len(labels) - n_overhand} ({100-accuracy:.1f}%)')
print(f'  Mean confidence:   {np.mean(confidences):.3f}')
print(f'{"="*60}')

---
## 9. Pipeline summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Cycle detection
ax = axes[0, 0]
t_plot = np.arange(len(mag0_smooth)) / fs
ax.plot(t_plot, mag0_smooth, 'k-', lw=0.5, alpha=0.7)
valid_peaks = [p for p in peaks0 if p < len(t_plot)]
ax.plot([t_plot[p] for p in valid_peaks],
        [mag0_smooth[p] for p in valid_peaks], 'rv', ms=3, alpha=0.6)
ax.set_xlabel('Time [s]')
ax.set_ylabel('||ω|| [rad/s]')
ax.set_title(f'Cycle detection: {len(paired0)} paired cycles')

# Panel 2: Template ω channels
ax = axes[0, 1]
x_phase = np.linspace(0, 1, template.shape[1])
for ch, color, lbl in [(3, '#5DCAA5', 'ωx'), (4, '#7F77DD', 'ωy'), (5, '#E24B4A', 'ωz')]:
    ax.plot(x_phase, template[ch], color=color, lw=2, label=f'template {lbl}')
ax.set_xlabel('Normalized cycle phase')
ax.set_ylabel('Angular velocity [rad/s]')
ax.set_title('Overhand template (D0 ω channels)')
ax.legend(fontsize=8)

# Panel 3: PCA projection
ax = axes[1, 0]
pca_vis = PCA(n_components=2)
X_pca = pca_vis.fit_transform(StandardScaler().fit_transform(X))
colors = ['#7F77DD' if l == 'overhand' else '#E24B4A' for l in labels]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6, s=20)
ax.set_xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Feature space (PCA projection)')

# Panel 4: Confidence timeline
ax = axes[1, 1]
bar_colors = ['#7F77DD' if l == 'overhand' else '#E24B4A' for l in labels]
ax.bar(range(len(confidences)), confidences, color=bar_colors, alpha=0.7)
ax.set_xlabel('Cycle index')
ax.set_ylabel('Confidence')
ax.set_title(f'Classification: {n_overhand}/{len(labels)} overhand ({accuracy:.1f}%)')

plt.suptitle('Rope Flow Pipeline V02 — Overhand Detection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Feature analysis

In [ ]:
# Top features by coefficient of variation
feat_cv = np.std(X, axis=0) / (np.abs(np.mean(X, axis=0)) + 1e-8)
top_idx = np.argsort(feat_cv)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top_idx)), feat_cv[top_idx][::-1], color='#7F77DD', alpha=0.8)
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels([ALL_FEAT_NAMES[i] for i in top_idx][::-1], fontsize=9)
ax.set_xlabel('Coefficient of variation')
ax.set_title('Top 15 most variable features across overhand cycles')
plt.tight_layout()
plt.show()

# Feature tier contribution
tier_slices = {
    'Physics (12D)': slice(0, 12),
    'Fourier (16D)': slice(12, 12 + FOURIER_DIM),
    'SVD (8D)':      slice(12 + FOURIER_DIM, 12 + FOURIER_DIM + SVD_DIM),
    'Template (2D)': slice(12 + FOURIER_DIM + SVD_DIM, N_FEAT),
}
print('Feature tier variance contribution:')
total_var = np.sum(np.var(X, axis=0))
for name, sl in tier_slices.items():
    tier_var = np.sum(np.var(X[:, sl], axis=0))
    print(f'  {name:<20s} {tier_var/total_var*100:6.1f}%')

---
## 11. Multi-class training & cross-validation

Updated for **2-class classification**: `overhand` vs `sneak_overhand`.

Key additions over V01:
- **DMD eigenvalue features** (9D) — dynamical fingerprint per cycle
- **Subject-invariant features** (9D) — D1/D0 ratios, skewness, kurtosis, inter-hand coupling
- **Per-session z-scoring** — removes subject-specific baseline before classification
- **GBM classifier** — handles feature interactions better than RF for small datasets
- **Session-level majority vote** — final verdict from per-cycle predictions

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report, f1_score
from collections import Counter

SESSIONS = [
    ('data/processed/20260303_174948_overhand_oli_device0_processed.csv',
     'data/processed/20260303_174948_overhand_oli_device1_processed.csv',
     'overhand', 'overhand_oli'),
    ('data/processed/20260303_175338_overhand_jo_device0_processed.csv',
     'data/processed/20260303_175338_overhand_jo_device1_processed.csv',
     'overhand', 'overhand_jo'),
    ('data/processed/20260303_182528_sneak_overhand_jo_device0_processed.csv',
     'data/processed/20260303_182528_sneak_overhand_jo_device1_processed.csv',
     'sneak_overhand', 'sneak_overhand_jo'),
    ('data/processed/20260303_181059_sneak_overhand_oli_device0_processed.csv',
     'data/processed/20260303_181059_sneak_overhand_oli_device1_processed.csv',
     'sneak_overhand', 'sneak_overhand_oli'),
]

### 11.1 Leave-one-session-out cross-validation

In [ ]:
print('LEAVE-ONE-SESSION-OUT CROSS-VALIDATION')
print('=' * 60)

loso_results = []
for test_idx in range(len(SESSIONS)):
    train_data = [(s[0], s[1], s[2]) for i, s in enumerate(SESSIONS) if i != test_idx]
    test_d0, test_d1, test_label, test_name = SESSIONS[test_idx]
    
    model = train_multi_session(train_data, verbose=False)
    result = classify_session(test_d0, test_d1, model, verbose=False)
    
    correct = result['verdict'] == test_label
    loso_results.append((test_name, test_label, result['verdict'], result['vote'], correct))
    status = 'CORRECT' if correct else 'WRONG'
    print(f'{test_name:<25s} GT={test_label:<16s} -> {result["verdict"]:<16s} {result["vote"]} [{status}]')

n_correct = sum(1 for r in loso_results if r[4])
print(f'\nSession-level accuracy: {n_correct}/{len(loso_results)} ({n_correct/len(loso_results)*100:.0f}%)')

### 11.2 Full model (all sessions)

In [ ]:
all_data = [(s[0], s[1], s[2]) for s in SESSIONS]
model_full = train_multi_session(all_data, verbose=True)

print(f'\nSelf-test (should be 100%):')
for d0, d1, label, name in SESSIONS:
    result = classify_session(d0, d1, model_full, verbose=False)
    status = 'CORRECT' if result['verdict'] == label else 'WRONG'
    print(f'  {name:<25s} GT={label:<16s} -> {result["verdict"]:<16s} [{status}]')

### 11.3 Feature importance

In [ ]:
# Feature importance from the full GBM model
imp = model_full['clf'].feature_importances_
feat_names = model_full['feature_names']
top = np.argsort(imp)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top)), imp[top][::-1], color='#7F77DD', alpha=0.8)
ax.set_yticks(range(len(top)))
ax.set_yticklabels([feat_names[i] for i in top][::-1], fontsize=9)
ax.set_xlabel('Feature importance (GBM)')
ax.set_title('Top 15 features for overhand vs sneak_overhand')
plt.tight_layout()
plt.show()

# Feature tier breakdown
tier_ranges = {
    'Physics (12D)': (0, 12),
    'Fourier (16D)': (12, 28),
    'SVD (8D)': (28, 36),
    'Template (2D)': (36, 38),
    'DMD (9D)': (38, 47),
    'Subject-inv (9D)': (47, 56),
}
print('Feature tier importance:')
for name, (lo, hi) in tier_ranges.items():
    tier_imp = np.sum(imp[lo:hi])
    print(f'  {name:<25s} {tier_imp:.3f} ({tier_imp/np.sum(imp)*100:.1f}%)')